# 📚 01 — The Knowledge Base and the RAG Agent

This course evaluates one support assistant for **Pinecrest Outdoor Supply**, a fictional
outdoor-gear store, from every angle. This first notebook builds the half of it that answers
questions from the store's support articles: a **RAG agent** (Retrieval-Augmented Generation).

## Learning Objectives
In this notebook, you will learn:
1. **Chunking with stable IDs** - why every piece of text needs an ID that never changes
2. **Indexing and retrieval** - how a question finds the closest chunks in a vector store
3. **Grounded answers** - an answer that cites its sources, or says "I don't know"
4. **Traces** - how LangSmith records each step, which is what evaluation later reads

## Prerequisites
- The course environment and kernel (see this folder's `README.md`)
- `OPENAI_API_KEY` and a LangSmith API key in the `.env` file at the repository root
- No earlier notebook: this is where the course starts

---
## 🗺️ The idea

A RAG agent makes two separate decisions, and evaluation scores each one on its own:

```
 question ──► RETRIEVE: which chunks are relevant? ──► GENERATE: answer from those chunks only
                 │                                         │
                 ▼                                         ▼
      scored in notebook 04                       scored in notebook 05
   (did we find the right chunks?)          (did the answer stay faithful to them?)
```

If the final answer is wrong, these two scores tell you *where* it went wrong.

---
## ⚙️ Part 0: Setup

Load the API keys and switch on LangSmith tracing. Every run of the agent becomes a **trace**:
a record of each step, its input, its output, its time and its cost.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: keys, tracing, models, paths
# ============================================================================
import os
import json
import glob
from typing import TypedDict

from dotenv import load_dotenv
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

# The keys live in the .env file at the repository root, two folders up.
load_dotenv(os.path.join("..", "..", ".env"))

# Record every agent run in LangSmith, in this course's own project,
# so traces from other courses don't mix in.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "end-to-end-eval-tutorial"

APP_MODEL = "gpt-4o-mini"                   # the model inside the agent
EMBEDDING_MODEL = "text-embedding-3-small"  # turns text into vectors for search

KB_DIR = os.path.join("data", "kb")              # the support articles
CHROMA_DIR = os.path.join("data", "chroma_db")   # the search index this notebook builds

print("✅ Setup complete")
print("🔑 OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("🔑 LangSmith key set:", bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")))
print("📈 Traces go to LangSmith project:", os.environ["LANGSMITH_PROJECT"])

✅ Setup complete
🔑 OPENAI_API_KEY set: True
🔑 LangSmith key set: True
📈 Traces go to LangSmith project: end-to-end-eval-tutorial


---
## 📄 Part 1: The knowledge base

Nine short support articles in `data/kb/`, one markdown file each: returns, refunds,
shipping, orders, warranty, privacy, loyalty points, escalation rules and product care.

In [2]:
# ============================================================================
# LOAD THE KNOWLEDGE BASE: one markdown file per support article
# ============================================================================
documents = []
for path in sorted(glob.glob(os.path.join(KB_DIR, "*.md"))):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    doc_id = os.path.basename(path).replace(".md", "")   # e.g. "returns-policy"
    documents.append({"doc_id": doc_id, "text": text})

print(f"📄 Loaded {len(documents)} articles:")
for doc in documents:
    title = doc["text"].splitlines()[0].lstrip("# ")
    print(f"   {doc['doc_id']:<24} {title}")

📄 Loaded 9 articles:
   account-and-privacy      Account and Privacy
   contact-and-escalation   Contacting a Human Agent
   loyalty-program          Trailpoints Loyalty Program
   orders                   Orders
   product-care             Product Care
   refunds                  Refunds
   returns-policy           Returns Policy
   shipping                 Shipping
   warranty                 Warranty


### 1.1 🧩 Chunking with stable IDs

Each `## ` section of an article becomes one **chunk**: a piece small enough to retrieve on its
own. Each chunk gets an ID built from its article and position, such as `returns-policy-02`.

> **Why this matters for evaluation:** notebook 03 labels every test question with the chunk
> IDs that answer it, and notebook 04 checks whether retrieval found those IDs. If IDs changed
> between runs, those labels would silently point at nothing.

In [3]:
# ============================================================================
# CHUNKING: one chunk per "## " section, with an ID that never changes
# ============================================================================
chunks = []
for doc in documents:
    title = doc["text"].splitlines()[0].lstrip("# ")
    sections = doc["text"].split("\n## ")[1:]      # item 0 is only the "# Title" line
    for number, section in enumerate(sections, start=1):
        lines = section.splitlines()
        heading = lines[0]
        body = "\n".join(lines[1:]).strip()
        chunks.append({
            "chunk_id": f"{doc['doc_id']}-{number:02d}",   # e.g. "returns-policy-02"
            "doc_id": doc["doc_id"],
            # The article title and heading stay in the text, so a chunk still
            # makes sense when the model reads it on its own.
            "text": f"{title} — {heading}\n{body}",
        })

# Save the chunk list: notebook 03 reads it to label the golden set.
with open(os.path.join("data", "kb_chunks.json"), "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"🧩 {len(chunks)} chunks from {len(documents)} articles → data/kb_chunks.json")
example = [chunk for chunk in chunks if chunk["chunk_id"] == "returns-policy-02"][0]
print("\nExample chunk:")
print("📄", example["chunk_id"])
print(example["text"])

🧩 31 chunks from 9 articles → data/kb_chunks.json

Example chunk:
📄 returns-policy-02
Returns Policy — Sale and clearance items
Items bought on sale or from the clearance section can be returned within 14 days of delivery.
After 14 days, sale items are final sale and cannot be returned or refunded. Sale items that
arrive damaged or defective are covered by the warranty instead (see the warranty policy).


---
## 🔍 Part 2: Indexing and retrieval

An **embedding** turns text into a list of numbers, so that texts with similar meaning get
similar numbers. Chroma stores one embedding per chunk. To answer a question, we embed the
question and fetch the chunks whose embeddings are closest.

In [4]:
# ============================================================================
# INDEXING: embed every chunk and store it in Chroma
# ============================================================================
# 💰 One call to the OpenAI embeddings API for all chunks (well under $0.01).
embedding_function = OpenAIEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    api_key_env_var="OPENAI_API_KEY",   # Chroma would otherwise look for CHROMA_OPENAI_API_KEY
)
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Rebuild from scratch each time, so re-running never leaves stale chunks behind.
existing_names = [c.name for c in chroma_client.list_collections()]
if "support_kb" in existing_names:
    chroma_client.delete_collection("support_kb")

collection = chroma_client.create_collection(
    name="support_kb",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"},   # measure closeness by cosine distance
)
collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[{"doc_id": chunk["doc_id"]} for chunk in chunks],
)
print(f"✅ Indexed {collection.count()} chunks into data/chroma_db/")

✅ Indexed 31 chunks into data/chroma_db/


In [5]:
# ============================================================================
# RETRIEVAL: find the chunks closest in meaning to a question
# ============================================================================
TOP_K = 4   # chunks handed to the model; notebook 04 measures recall at this k

def retrieve(question, k=TOP_K):
    """Return the k chunks closest to the question, closest first."""
    results = collection.query(query_texts=[question], n_results=k)
    hits = []
    for chunk_id, text, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
        hits.append({"chunk_id": chunk_id, "text": text, "distance": round(distance, 3)})
    return hits

# 💰 One small embedding call for the question.
# Cosine distance: 0 means the same meaning; bigger means less related.
for hit in retrieve("Can I return a sale item after three weeks?"):
    print(f"🔍 {hit['chunk_id']:<28} distance = {hit['distance']}")

🔍 returns-policy-02            distance = 0.423
🔍 returns-policy-01            distance = 0.484
🔍 refunds-01                   distance = 0.574
🔍 returns-policy-03            distance = 0.585


---
## 🤖 Part 3: Grounded answers

The model may answer **only** from the retrieved chunks, must cite the chunk behind each fact,
and must say so when the chunks don't contain the answer.

> **Note:** "I don't know" is a correct answer when the articles don't cover the question. A
> support assistant that invents a refund rule is worse than one that declines. The golden set
> in notebook 03 includes questions built to test exactly this.

In [6]:
# ============================================================================
# GENERATION: answer only from the retrieved chunks, and cite them
# ============================================================================
llm = ChatOpenAI(model=APP_MODEL, temperature=0)   # temperature 0: as repeatable as possible

RAG_PROMPT = """You are the support assistant for Pinecrest Outdoor Supply.
Answer the customer's question using ONLY the support articles below.
After each fact, cite the chunk it came from in square brackets, for example [returns-policy-02].
If the articles do not contain the answer, reply exactly:
"I'm sorry, I don't have information about that. I can connect you with a human agent."

Support articles:
{context}"""

def generate_answer(question, chunk_ids, contexts):
    # Label each chunk with its ID, so the model can cite it.
    labelled_chunks = []
    for chunk_id, text in zip(chunk_ids, contexts):
        labelled_chunks.append(f"[{chunk_id}]\n{text}")
    messages = [
        SystemMessage(content=RAG_PROMPT.format(context="\n\n".join(labelled_chunks))),
        HumanMessage(content=question),
    ]
    return llm.invoke(messages).content

print("✅ Answer generator ready, model:", APP_MODEL)

✅ Answer generator ready, model: gpt-4o-mini


---
## 🔗 Part 4: The RAG agent as a graph

LangGraph runs the two steps as two **nodes** that share a **state** (a dictionary every node
can read and add to). Two nodes, not one function, so each step appears separately in the
trace, and later notebooks can score each one on its own.

In [7]:
# ============================================================================
# THE RAG AGENT: START → retrieve → generate → END
# ============================================================================
# LangGraph needs the shape of the state written down as a TypedDict.
# Each field is something a later notebook evaluates.
class RAGState(TypedDict):
    question: str
    retrieved_chunk_ids: list   # which chunks were found      → retrieval metrics, notebook 04
    contexts: list              # the text of those chunks     → faithfulness, notebook 05
    answer: str                 # what the customer sees       → answer quality, notebook 05

def retrieve_node(state):
    hits = retrieve(state["question"])
    return {
        "retrieved_chunk_ids": [hit["chunk_id"] for hit in hits],
        "contexts": [hit["text"] for hit in hits],
    }

def generate_node(state):
    answer = generate_answer(state["question"], state["retrieved_chunk_ids"], state["contexts"])
    return {"answer": answer}

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
rag_agent = builder.compile()

print("✅ RAG agent compiled: START → retrieve → generate → END")

✅ RAG agent compiled: START → retrieve → generate → END


---
## 🧪 Part 5: Try it

Three questions, each a different kind of test. They preview the **slices** of the golden set
in notebook 03: every metric later in the course is reported per slice, because an average
over all questions hides which kind of question fails.

In [8]:
# ============================================================================
# TRY IT: three questions, three kinds of difficulty
# ============================================================================
# 💰 Three agent runs with gpt-4o-mini (about one cent in total).
questions = [
    "How long does express shipping take?",                           # simple: one chunk answers it
    "I bought a jacket on sale 20 days ago. Can I still return it?",  # multi-hop: sale rule + window
    "Do you sell fishing licences?",                                  # out of scope: should decline
]

for question in questions:
    result = rag_agent.invoke(
        {"question": question},
        config={"run_name": "rag-agent", "tags": ["notebook-01"]},   # labels for the LangSmith trace
    )
    print("❓", question)
    print("🔍 retrieved:", result["retrieved_chunk_ids"])
    print("🤖", result["answer"])
    print("-" * 90)

print("📈 Open LangSmith → project 'end-to-end-eval-tutorial' to see each run as a trace.")

❓ How long does express shipping take?
🔍 retrieved: ['shipping-01', 'shipping-02', 'shipping-03', 'shipping-04']
🤖 Express shipping takes 1 to 2 business days [shipping-01].
------------------------------------------------------------------------------------------


❓ I bought a jacket on sale 20 days ago. Can I still return it?
🔍 retrieved: ['returns-policy-02', 'returns-policy-01', 'refunds-01', 'product-care-02']
🤖 No, you cannot return the jacket because items bought on sale can only be returned within 14 days of delivery. Since it has been 20 days, the sale item is considered final sale and cannot be returned or refunded [returns-policy-02].
------------------------------------------------------------------------------------------


❓ Do you sell fishing licences?
🔍 retrieved: ['loyalty-program-01', 'warranty-02', 'shipping-02', 'returns-policy-02']
🤖 I'm sorry, I don't have information about that. I can connect you with a human agent.
------------------------------------------------------------------------------------------
📈 Open LangSmith → project 'end-to-end-eval-tutorial' to see each run as a trace.


### 5.1 📈 What to look at in LangSmith

Open one trace. It shows the two nodes, `retrieve` and `generate`, each with its input, output
and duration, and the model call inside `generate` with its prompt, tokens and cost. That trace
is the raw material of evaluation: every metric in this course scores something recorded in it.

| Recorded in the trace | Evaluated in |
|---|---|
| `retrieved_chunk_ids` | Notebook 04: did retrieval find the right chunks? |
| `contexts` + `answer` | Notebook 05: is the answer faithful to the chunks? |
| Tokens, cost, latency | Notebook 12: what does each answer cost? |

---
## 📝 Summary

In this notebook, we learned:

### 1. Chunking
- **Stable IDs**: `returns-policy-02` names the same text on every run, so test labels stay valid
- **Saved chunk list**: `data/kb_chunks.json` is what notebook 03 labels the golden set against

### 2. Retrieval and generation
- **Two decisions**: what to retrieve and what to say are separate steps, scored separately
- **Grounded answers**: cite every fact; decline when the articles don't cover the question

### 3. Traces
- **LangSmith records each node**: inputs, outputs, tokens, cost and time per step

### Next Steps
- Notebook 02 adds tools (order lookup, refunds, tickets), an account agent and a supervisor that
  routes each question, turning this RAG agent into the full support assistant